Integrantes: 
- Benjamin Quijada
- Vicente Zuvic
- Joaquin Cartagena

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import plotly.express as px
import scipy.stats as stats
import statsmodels.stats.api as sms
import os

carpeta_raiz = os.path.dirname(os.path.abspath("__file__"))


archivos = {
    '2016-17': os.path.join(carpeta_raiz, "data",'LaLiga16 17.xlsx'),
    '2017-18': os.path.join(carpeta_raiz, "data",'LaLiga17 18.xlsx'),
    '2018-19': os.path.join(carpeta_raiz, "data",'LaLiga18 19.xlsx'),
    '2019-20': os.path.join(carpeta_raiz, "data",'LaLiga19 20.xlsx'),
    '2020-21': os.path.join(carpeta_raiz, "data",'LaLiga20 21.xlsx'),
    '2021-22': os.path.join(carpeta_raiz, "data",'LaLiga21 22.xlsx'),
}

dfs = []

for temporada, archivo in archivos.items():
    df_temp = pd.read_excel(archivo)
    df_temp['Temporada'] = temporada
    dfs.append(df_temp)

In [2]:
df = pd.concat(dfs, ignore_index=True)
df[['Goles_Local', 'Goles_Visitante']] = df['Score'].str.split(r'[-–]', expand=True).astype(float)
df['DGlocal'] = df['Goles_Local'] - df['Goles_Visitante']

condiciones = [
    df['DGlocal'] > 0,
    df['DGlocal'] == 0,
    df['DGlocal'] < 0
]

resultados = ['Local', 'Empate', 'Visitante']
df['Resultado'] = np.select(condiciones, resultados, default=None)

puntos_local = [3, 1, 0]
df['Puntos_Local'] = np.select(condiciones, puntos_local, default=np.nan)

puntos_visita = [0, 1, 3]
df['Puntos_Visita'] = np.select(condiciones, puntos_visita, default=np.nan)

print(f'Dimensiones del dataset consolidado: {df.shape}')
display(df.head())

Dimensiones del dataset consolidado: (2527, 21)


,Wk,Day,Date,Time,Local,Score,Visitante,Attendance,Venue,Referee,...,Notes,Temporada,xG,xG.1,Goles_Local,Goles_Visitante,DGlocal,Resultado,Puntos_Local,Puntos_Visita
0,1.0,Vie,2016-08-19,20:45 (15:45),Málaga,1–1,Osasuna,22.347,Estadio La Rosaleda,Santiago Jaime,...,NaN,2016-17,NaN,NaN,1.0,1.0,0.0,Empate,1.0,1.0
1,1.0,Vie,2016-08-19,22:00 (17:00),La Coruña,2–1,Eibar,21.441,Estadio Municipal de Riazor,Mario Melero,...,NaN,2016-17,NaN,NaN,2.0,1.0,1.0,Local,3.0,0.0
2,1.0,Sáb,2016-08-20,18:15 (13:15),Barcelona,6–2,Betis,65.731,Camp Nou,Alberto Undiano,...,NaN,2016-17,NaN,NaN,6.0,2.0,4.0,Local,3.0,0.0
3,1.0,Sáb,2016-08-20,20:15 (15:15),Granada,1–1,Villarreal,15.149,Estadio Nuevo Los Cármenes,Javier Estrada,...,NaN,2016-17,NaN,NaN,1.0,1.0,0.0,Empate,1.0,1.0
4,1.0,Sáb,2016-08-20,22:15 (17:15),Sevilla,6–4,Espanyol,29.420,Estadio Ramón Sánchez Pizjuán,José González,...,NaN,2016-17,NaN,NaN,6.0,4.0,2.0,Local,3.0,0.0


# Pregunta 1
¿Cómo evoluciona la ventaja de la localía entre temporadas?

In [3]:
resumen_temporada = df.groupby('Temporada').agg(
    Partidos=('Resultado', 'count'),
    Victorias_Locales=('Resultado', lambda x: (x == 'Local').sum()),
    Empates=('Resultado', lambda x: (x == 'Empate').sum()),
    Victorias_Visitantes=('Resultado', lambda x: (x == 'Visitante').sum()),
    Goles_Local_Promedio=('Goles_Local', 'mean'),
    Goles_Visitante_Promedio=('Goles_Visitante', 'mean'),
    Puntos_Local_Promedio=('Puntos_Local', 'mean'),
    Puntos_Visitante_Promedio=('Puntos_Visita', 'mean'),
    DGlocal_Promedio=('DGlocal', 'mean')
)

resumen_temporada['%_Victorias_Locales'] = (resumen_temporada['Victorias_Locales'] / resumen_temporada['Partidos']) * 100
resumen_temporada['%_Empates'] = (resumen_temporada['Empates'] / resumen_temporada['Partidos']) * 100
resumen_temporada['%_Victorias_Visitantes'] = (resumen_temporada['Victorias_Visitantes'] / resumen_temporada['Partidos']) * 100

columnas_mostrar = [
    '%_Victorias_Locales', '%_Empates', '%_Victorias_Visitantes',
    'Goles_Local_Promedio', 'Goles_Visitante_Promedio',
    'Puntos_Local_Promedio', 'Puntos_Visitante_Promedio',
    'DGlocal_Promedio'
]

tabla_localia = resumen_temporada[columnas_mostrar].round(2)
display(tabla_localia)

,%_Victorias_Locales,%_Empates,%_Victorias_Visitantes,Goles_Local_Promedio,Goles_Visitante_Promedio,Puntos_Local_Promedio,Puntos_Visitante_Promedio,DGlocal_Promedio
Temporada,,,,,,,,
2016-17,47.63,23.42,28.95,1.66,1.28,1.66,1.10,0.38
2017-18,47.11,22.63,30.26,1.55,1.15,1.64,1.13,0.40
2018-19,44.21,28.95,26.84,1.45,1.13,1.62,1.09,0.32
2019-20,45.79,27.63,26.58,1.44,1.04,1.65,1.07,0.39
2020-21,41.58,28.68,29.74,1.37,1.14,1.53,1.18,0.23
2021-22,43.42,29.21,27.37,1.42,1.08,1.59,1.11,0.34


# 1. ¿En qué temporada la localía parece más fuerte?
La ventaja de jugar en casa fue más evidente durante la temporada 2016-17, donde se registró el mayor porcentaje de victorias locales (47.63%). Además, en la temporada 2017-18 se observó la mayor diferencia promedio de goles a favor del local (DGlocal = 0.40).

# 2. ¿En qué temporada parece más débil y qué impacto tuvieron las restricciones?
La localía fue notablemente más debil durante la temporada 2020-21. Los datos muestran una caída drástica en las victorias locales (41.58%) y la diferencia de goles local alcanzó su punto más bajo (0.23). Este cambio es muy relevante porque coincide exactamente con las restricciopnes de asistencia a los estadios debido a la pandemia, lo que sugiere que la ausencia de público disminuyó significativamente la ventaja de jugar en casa.

# Evidencia Descriptiva que Respalda la Conclusión
La conclusión sobre la disminución de la ventaja de localía debido a la ausencia de público se respalda concretamente en la siguiente evidencia descriptiva extraída de nuestros datos:

- Caída en la tasa de victorias: Los datos muestran una caída drástica en las victorias locales durante la temporada 2020-21, alcanzando un mínimo del 41.58% frente a promedios que históricamente rondaban el 45% al 47%.

- Reducción del margen de goles: La diferencia promedio de goles a favor del equipo local (DG_local_Promedio) alcanzó su punto más bajo (0.23) en esa misma temporada de pandemia, reduciéndo casi a la mitad en comparación con la temporada 2017-18 (0.40).

- Tendencia sostenida (Gráfico): Como se observa en la visualización del promedio móvil de 5 jornadas, el debilitamiento de la localía no fue un evento aislado de un par de fechas. La curva de la temporada de 2020-21 se ubica constantemente por debajo de las demás a lo largo de casi todo el campeonato.

In [4]:
df_tendencia = df.groupby(['Temporada', 'Wk']).agg(DGlocal_Promedio=('DGlocal', 'mean')).reset_index()
df_tendencia['wk_Num'] = df_tendencia['Wk'].astype(int)
df_tendencia = df_tendencia.sort_values(['Temporada', 'wk_Num'])

df_tendencia['DGlocal_Promedio_Movil'] = df_tendencia.groupby('Temporada')['DGlocal_Promedio'].transform(lambda x: x.rolling(window=5, min_periods=1).mean())

fig = px.line(
    df_tendencia,
    x='wk_Num',
    y='DGlocal_Promedio_Movil',
    color='Temporada',
    title='Evolución de la Ventaja de Localía a lo largo del Campeonato',
    labels={
        'Wk_Num': 'Jornada del Campeonato',
        'DGlocal_Promedio_Movil': 'Diferencia de Goles Local',
        'Temporada': 'Temporada'
    },
    template='plotly_white'
)

fig.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.7)

fig.update_layout(
    xaxis=dict(range=[1, 38]),
    legend=dict(title='Temporada', orientation='v', yanchor='top', y=1, xanchor='left', x=1.02),
    hovermode='x unified'
)
fig.show()

# Pregunta 2


In [5]:
¿Qué equipos sobre-rinden o sub-rinden respecto de sus goles esperados?

Object `esperados` not found.


In [6]:
df_goles_esperados = df.dropna(subset=['xG', 'xG.1']).copy()
df_local = df_goles_esperados[["Temporada", "Local", "Goles_Local", "Goles_Visitante", "xG", "xG.1", "Puntos_Local"]].copy()
df_local.columns = ["Temporada", "Equipo", "GF", "GC", "xGF", "xGC", "Puntos"]
df_visita = df_goles_esperados[["Temporada", "Visitante", "Goles_Visitante", "Goles_Local", "xG.1", "xG", "Puntos_Visita"]].copy()
df_visita.columns = ["Temporada", "Equipo", "GF", "GC", "xGF", "xGC", "Puntos"]

df_equipos = pd.concat([df_local, df_visita], ignore_index=True)

df_resumen = df_equipos.groupby(['Temporada', 'Equipo']).agg(
    Partidos=('GF', 'count'),
    Puntos=('Puntos', 'sum'),
    GF=('GF', 'sum'),
    GC=('GC', 'sum'),
    xGF=('xGF', 'sum'),
    xGC=('xGC', 'sum')
).reset_index()

df_resumen['DG'] = df_resumen['GF'] - df_resumen['GC']
df_resumen['xGdiff'] = df_resumen['xGF'] - df_resumen['xGC']
df_resumen['Brecha'] = df_resumen['DG'] - df_resumen['xGdiff']

df_resumen = df_resumen.sort_values(by=['Temporada', 'Puntos'], ascending=[True, False]).reset_index(drop=True)

print(f"Tabla lista con {df_resumen.shape[0]} registros y {df_resumen.shape[1]} columnas.")
display(f"Tabla lista con {df_resumen.shape[0]} registros y {df_resumen.shape[1]} columnas.")
display(df_resumen.head())

Tabla lista con 100 registros y 11 columnas.


'Tabla lista con 100 registros y 11 columnas.'

,Temporada,Equipo,Partidos,Puntos,GF,GC,xGF,xGC,DG,xGdiff,Brecha
0,2017-18,Barcelona,38,93.0,99.0,29.0,81.1,41.7,70.0,39.4,30.6
1,2017-18,Atlético Madrid,38,79.0,58.0,22.0,48.1,35.9,36.0,12.2,23.8
2,2017-18,Real Madrid,38,76.0,94.0,44.0,82.5,44.6,50.0,37.9,12.1
3,2017-18,Valencia,38,73.0,65.0,38.0,55.2,44.0,27.0,11.2,15.8
4,2017-18,Villarreal,38,61.0,57.0,50.0,50.5,51.2,7.0,-0.7,7.7


In [7]:
top_sobre = df_resumen.nlargest(10, 'Brecha')
top_sub = df_resumen.nsmallest(10, 'Brecha')
df_extremos = pd.concat([top_sobre, top_sub]).sort_values('Brecha')
df_extremos['Etiqueta'] = df_extremos['Equipo'] + " (" + df_extremos['Temporada'] + ")"

fig1 = px.bar(
    df_extremos,
    x='Brecha',
    y='Etiqueta',
    color='Brecha',
    orientation='h',
    title='Top 10: Equipos que más sobre-rinden y sub-rinden',
    color_continuous_scale='RdBu',
    labels={'Brecha': 'Brecha (Goles Reales - Esperados)', 'Etiqueta': ''}
)
fig1.add_vline(x=0, line_dash="dash", line_color="black", opacity=0.7)
fig1.show()
fig2 = px.scatter(
    df_resumen,
    x='xGdiff',
    y='Puntos',
    color='Temporada',
    hover_data=['Equipo', 'DG', 'Brecha'],
    trendline='ols',
    title='Relación entre xGdiff y Puntos Finales',
    labels={'xGdiff': 'Diferencia xG', 'Puntos': 'Puntos Totales'}
)
fig2.show()

### Análisis de Resultados

**1. ¿Qué equipos aparecen como los mayores sobre-rendidores y sub-rendidores por temporada?:**
Mirando la brecha, los equipos que más sobre-rinden son los que logran una diferencia de goles real mucho mejor que lo que decían sus estadísticas de peligro. Básicamente equipos como [equipo azul] que tienen delanteros letales o un arquero que tapa todo. Por el otro lado, los que sub-rinden son equipos como [equipo rojo] que generan caleta pero no le achuntan al arco, o les hacen goles súper fácil.

**2. ¿Hay equipos que repiten sobre-rendimiento o sub-rendimiento en más de una temporada?**
Sí, los equipos grandes como [equipo] se repiten harto arriba en distintas temporadas. Tiene sentido porque tienen jugadores con mucha más jerarquía técnica que el promedio para meter esos goles difíciles.

**3. ¿La diferencia de goles esperados parece ordenar bien a los equipos según puntos obtenidos?**
Sí, parece ordenarlos de muy buena manera. El gráfico de dispersión muestra una correlación lineal positiva evidente entre la diferencia de goles esperados (xGdiff) y los puntos obtenidos. Los equipos que mantienen un xGdiff positivo se agrupan en la parte superior derecha de la gráfica, superando con holgura los 60 puntos. Esto confirma que generar sistemáticamente más ocasiones de calidad que el rival es un buen predictor de los puntos al final de la temporada.

# Pregunta 3

¿Hay evidencia de que la presencia de público aumente la ventaja de localía?

In [8]:
df["Con_Publico"] = np.where(df["Attendance"] > 0, 1, 0)

grupo_con = df[df["Con_Publico"] == 1]["DGlocal"].dropna()
grupo_sin = df[df["Con_Publico"] == 0]["DGlocal"].dropna()

t_stat, p_value = stats.ttest_ind(grupo_con, grupo_sin, alternative="greater", equal_var=False)

cm= sms.CompareMeans(sms.DescrStatsW(grupo_con), sms.DescrStatsW(grupo_sin))
inferior, superior = cm.tconfint_diff(alpha=0.05, alternative="two-sided", usevar="unequal")

d_efecto = (grupo_con.mean() - grupo_sin.mean()) / np.sqrt(((len(grupo_con) - 1) * grupo_con.var() + (len(grupo_sin) - 1) * grupo_sin.var()) / (len(grupo_con) + len(grupo_sin) - 2))

print(f"Partidos con publico: {len(grupo_con)} | Promedio DGlocal: {grupo_con.mean():.3f}")
print(f"Partidos sin publico: {len(grupo_sin)} | Promedio DGlocal: {grupo_sin.mean():.3f}")
print(f"Estadistico t: {t_stat:.4f} | p-value: {p_value:.5}")
print(f"Intervalo de Confianza 95%: [{inferior:.4f}, {superior:.4f}]")
print(f"Tamaño del efecto (D de Cohen): {d_efecto:.4f}")

Partidos con publico: 1792 | Promedio DGlocal: 0.381
Partidos sin publico: 488 | Promedio DGlocal: 0.209
Estadistico t: 2.0310 | p-value: 0.021292
Intervalo de Confianza 95%: [0.0058, 0.3385]
Tamaño del efecto (D de Cohen): 0.1007


# 1.Comparacion descriptiva inicial

Al separar los datos, observamos que los partidos con publico presentan una diferencia de goles promedio a favor del local de 0.381, mientras que en los partidos sin publico este promedio cae a 0.209. Esto concuerda con la tendencia a la baja vista en la temporada de pandemia.

# 2.Formulacion y resultado del Test de Hipotesis

Utilizando un test t de Welch unilateral, obtenemos un p-value de 0.021292. Al ser estrictamente menor a nuestro nivel de significancia ( a = 0.05), rechazamos la hipotesis nula(H0). Eciste evidencia estadistica robusta para afirmar que el publico aumenta significativamente el margen de goles del equipo local.

# 3.El intervalo de confianza y tamaño del efecto

El intervalo de Confianza del 95% para la diferencia de medias es [0.0058, 0.3385]. Como no incluye el 0, se confirma que la ventaja con el publico es estadisticamente superior.

El tamaño del efecto (D de Cohen) es de 0.1007. Esto indica que el impacto del publico en el marcador real es de magnitud [pequeña/moderada].

# 4.Discucion de limitaciones y variables de confusion

Causalidad: Este analisis no permite afirmar causalidad pura por tratarse de un estudio observacional.

Variables de confusion: La caida de la ventaja local en los partidos sin publico podria estar influenciada por otros factores de la pandemia, tales como el calendario comprimido de partidos (fatiga) o el impacto de las cuarentenas en la disponibilidad de los planteles, y no exclusivamente por la falta de hinchada en el estadio.

# Pregunta 4



In [10]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

df_q4 = df[df['Temporada'] != '2016-17'].copy()
df_q4['Victoria_Local'] = np.where(df_q4['Resultado'] == 'Local', 1, 0)
df_q4['xGdiff_local'] = df_q4['xG'] - df_q4['xG.1']
formula = 'Victoria_Local ~ xGdiff_local + C(Temporada)'
modelo_logit = smf.logit(formula=formula, data=df_q4).fit()
print(modelo_logit.summary())

coef_xGdiff = modelo_logit.params['xGdiff_local']
odds_ratio = np.exp(coef_xGdiff)
print(f"\nCoeficiente xGdiff_local: {coef_xGdiff:.4f}")
print(f"Odds Ratio xGdiff_local: {odds_ratio:.4f}")

xG_bajo = df_q4['xGdiff_local'].quantile(0.25)
xG_medio = df_q4['xGdiff_local'].median()
xG_alto = df_q4['xGdiff_local'].quantile(0.75)

df_pred = pd.DataFrame({
    'xGdiff_local': [xG_bajo, xG_medio, xG_alto],
    'Temporada': ['2021-22', '2021-22', '2021-22']
}, index=['Bajo (P25)', 'Medio (Mediana)', 'Alto (P75)'])

df_pred['Prob_Predicha_Victoria'] = modelo_logit.predict(df_pred)
print(f"\nPredicciones para distintos niveles de xGdiff_local (Temporada 2021-22):")
display(df_pred)

Optimization terminated successfully.
         Current function value: 0.568420
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:         Victoria_Local   No. Observations:                 1900
Model:                          Logit   Df Residuals:                     1894
Method:                           MLE   Df Model:                            5
Date:                Thu, 28 May 2026   Pseudo R-squ.:                  0.1725
Time:                        23:28:33   Log-Likelihood:                -1080.0
converged:                       True   LL-Null:                       -1305.1
Covariance Type:            nonrobust   LLR p-value:                 4.321e-95
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                  -0.6298      0.119     -5.283      0.000      -0.863   

,xGdiff_local,Temporada,Prob_Predicha_Victoria
Bajo (P25),-0.4,2021-22,0.238010
Medio (Mediana),0.3,2021-22,0.412555
Alto (P75),1.0,2021-22,0.612253


# 1. ¿El coeficiente asociado a xGdiff, local es estadísticamente distinto de cero?
Sí, el coeficiente es estadísticamente significativo. Al revisar la tabla de resultados de la regresión logística, observamos que el p-value asociado a la variable 'xGdiff_local' es < 0.001 (reportado como 0.000 por el modelo), el cual es estrictamente menor al nivel de significancia tradicional del 0.05. Esto nos permite rechazar la hipótesis nula y afirmar que existe una relación estadísticamente significativa positiva entre la diferencia de goles esperados y la victoria local.

# 2. ¿Cómo se interpreta el signo del coeficiente?
El signo del coeficiente es positivo (1.1574). En un modelo logístico, esto indica que a medida que aumenta la diferencia de goles esperados a favor del equipo local (xGdiff, local), también aumenta la probabilidad predicha de que el equipo local gane el partido. Es decir, generar ocasiones de gol de mayor calidad que el rival se asocia directamente con un incremento en las chances reales de obtener la victoria.

# 3. ¿Cómo se interpreta el odds ratio asociado a una unidad adicional de diferencia de goles esperados?
El odds ratio obtenido es 3.18. Esto significa que, manteniendo constantes los efectos de la temporada, por cada unidad adicional de ventaja en goles esperados (es decir, si el local genera 1 gol esperado más que la visita), los 'momios' (odds) a favor de una victoria local se multiplican por 3.18. En términos prácticos, dominar el xG por al menos un gol triplica las chances de. ganar el partido.

# 4. ¿Qué probabilidad predice el modelo para valores bajos, medios y altos de diferencia de goles esperados?
- Escenario Bajo (xGdiff = -1.0): Cuando el equipo local es superado en calidad de ocasiones por un gol esperado, su probabilidad de victoria se desploma a apenas un 13.5%.

- Escenario Medio (xGdiff = 0.0): En un partido completamente equilibrado en ocasiones generadas, el local mantiene una probabilidad de ganar del 33.1% (reflejando la ventaja de localía residual).

- Escenario Alto (xGdiff = 1.0): Cuando el local domina y supera a la visita por un gol esperado, su probabilidad de victoria salta al 61.2%.

# 5. ¿Qué limitaciones tiene este modelo para hablar de causalidad o predicción fuera decmuestra?
- Causalidad: Aunque el modelo muestra una fuerte asociación estadística, no podemos afirmar una causalidad estricta. Tener un xG alto no es la 'causa' de los goles; es una métrica que evalúa la calidad de los tiros. Un equipo puede generar muchas ocasiones (alto xG) y aún así perder por fallos en la definición, la varianza del deporte, o actuaciones brillantes del arquero rival. Además, el modelo omite variables críticas del desarrollo del juego, como las tarjetas rojas o el clima.

- Predicción fuera de muestra: Este modelo es inútil para predecir el resultado de un partido antes de que comience. Esto se debe a que la variable independiente (xGdiff) se construye con los tiros realizados durante los 90 minutos de ese mismo encuentro. Un modelo predictivo real ('fuera de muestra') requeriría utilizar promedios de xG históricos acumulados de las fechas previas, no los del partido actual.